In [12]:
from dotenv import load_dotenv,find_dotenv
import os
from langgraph.graph import StateGraph,END,START
from langchain_openai import AzureChatOpenAI
from typing import TypedDict
from operator import add

load_dotenv(find_dotenv(),override=True)

class AgentInputState(TypedDict):
    question: str

class AgentPrivateState(TypedDict):
    llm_calls:int

class AgentOutputState(TypedDict):
    answer: str

class OverAllState(AgentInputState,AgentPrivateState,AgentOutputState):
    pass

llm=AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("API_VERSION"))

In [13]:
def call_model(state:OverAllState) -> OverAllState:   
    question = (state["question"])
    llm_call=state.get("llm_calls",0)
    state["llm_calls"]=llm_call+1
    # print(f"LLM call - {state['llm_calls']} ")
    res=llm.invoke(question)
    state["answer"]=res.content
    return state

In [15]:
graph=StateGraph(OverAllState,input=AgentInputState,output=AgentOutputState)
graph.add_node("agent",call_model)

graph.add_edge(START,"agent")
graph.add_edge("agent",END) 

app=graph.compile()
res=app.invoke({"question":"What is the capital of France?"})
res


{'answer': 'The capital of France is **Paris**.'}